In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Signal-Space Image Generation: Final Paper Supporting Runs

This notebook produces the experimental results that fill the
remaining gaps in the paper draft. The runs are ordered by priority
so the most paper-critical numbers come first.

## What this notebook does

**P1 (must-have).** Multi-seed verification of the 48x48 scaling
result. The paper currently reports the scaling experiment with
a single seed; replacing the single-seed entry in Table II with a
3-seed mean and standard deviation makes the scaling claim
defensible.

**P2 (important).** Generation samples for a side-by-side qualitative
figure (V5 path-only vs PixelCNN++ at matched epochs). The paper has
no samples figure yet; reviewers expect one.

**P3 (nice-to-have).** Complete the 32x32 ablation grid by adding
multi-seed `seq only` and `neither` configurations. The paper
currently dismisses these as "single seed only"; with the data they
become a complete 4-cell ablation matrix.

## What this notebook does NOT do

It does not re-run V5 itself or PixelCNN++ from scratch. Both have
already been multi-seed-verified at 32x32 under the established
protocol; the previous runs are the source of truth and are
referenced by the paper as-is. This notebook only adds the missing
pieces.

## Compute budget

Approximately 9 hours of T4 time for all three priorities. Each
priority can be run in its own session.

| Priority | Description | Runs | T4 hours |
|---|---|---|---|
| P1 | 48x48 multi-seed | 6 (2 configs x 3 seeds, 15 epochs) | ~6 |
| P2 | Generation samples | 2 (one per model) | ~0.5 |
| P3 | Complete 32x32 grid | 6 (2 configs x 3 seeds, 30 epochs) | ~3 |


## 1. Environment setup

In [ ]:
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q datasets

In [ ]:
import os

if os.path.isdir("/kaggle/working"):
    WORK_DIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK_DIR = "/content"
else:
    WORK_DIR = os.getcwd()

hf_token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN") or hf_token
except Exception:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN") or hf_token
    except Exception:
        pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
print(f"WORK_DIR = {WORK_DIR}")

In [ ]:
import math
import copy
import statistics
from dataclasses import dataclass, field
from typing import Optional, Tuple, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

## 2. V5 architecture (unchanged)

Sections 2--12 reproduce the V5 architecture verbatim. They are
copied here so the notebook is self-contained; nothing in them
should change between runs.

In [ ]:
@dataclass
class Config:
    image_size: int = 32
    channels: int = 3
    samples_per_pixel: int = 2
    flyback_frac: float = 0.08
    beam_sigma: float = 0.75

    clip_dim: int = 512
    cond_dim: int = 256
    d_model: int = 256
    n_heads: int = 4
    n_layers: int = 6
    ff_mult: int = 4
    dropout: float = 0.1
    n_mixtures: int = 5

    batch_size: int = 24
    epochs: int = 30
    lr: float = 3e-4
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    warmup_steps: int = 500
    ss_max: float = 0.25

    use_path_pos_enc: bool = True
    use_seq_pos_enc: bool = True
    use_flyback_mask: bool = True
    use_film: bool = True

    seed: int = 0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
print(f"device: {cfg.device}")

## 3. Scan path, renderer, dataset

In [ ]:
def generate_raster_path(w, h, samples_per_pixel, flyback_frac):
    samples_per_row = w * samples_per_pixel
    flyback_samples = max(int(samples_per_row * flyback_frac), 1)
    xs, ys, on = [], [], []
    for row in range(h):
        y = row + 0.5
        for i in range(samples_per_row):
            x = (i / max(samples_per_row - 1, 1)) * (w - 1)
            xs.append(x); ys.append(y); on.append(True)
        for i in range(flyback_samples):
            t = (i + 1) / flyback_samples
            x = (1 - t) * (w - 1)
            xs.append(x); ys.append(y); on.append(False)
    path = np.stack([np.array(xs, np.float32), np.array(ys, np.float32)], axis=1)
    beam_on = np.array(on, dtype=bool)
    return path, beam_on


class CRTRenderer(nn.Module):
    def __init__(self, path, beam_on, image_size, sigma):
        super().__init__()
        self.image_size = image_size
        self.sigma = sigma
        self.register_buffer("path_x", torch.from_numpy(path[:, 0]).float())
        self.register_buffer("path_y", torch.from_numpy(path[:, 1]).float())
        self.register_buffer("beam_on_f",
                             torch.from_numpy(beam_on.astype(np.float32)))
        ys, xs = torch.meshgrid(
            torch.arange(image_size, dtype=torch.float32),
            torch.arange(image_size, dtype=torch.float32),
            indexing="ij",
        )
        self.register_buffer("grid_x", xs)
        self.register_buffer("grid_y", ys)
        samples_per_pixel = beam_on.sum() / image_size ** 2
        gain = samples_per_pixel * 2.0 * math.pi * sigma * sigma
        self.register_buffer("gain", torch.tensor(max(gain, 1e-3)))

    def forward(self, signal):
        b, n, c = signal.shape
        h = w = self.image_size
        sig = signal * self.beam_on_f.view(1, n, 1)
        dx = self.grid_x.view(1, h, w) - self.path_x.view(n, 1, 1)
        dy = self.grid_y.view(1, h, w) - self.path_y.view(n, 1, 1)
        weights = torch.exp(-(dx * dx + dy * dy) / (2.0 * self.sigma ** 2))
        img = torch.einsum("bnc,nhw->bchw", sig, weights) / self.gain
        return img.clamp(0.0, 1.0)


def image_to_signal(image, path_t, beam_on_t):
    c, h, w = image.shape
    xs = 2.0 * path_t[:, 0] / (w - 1) - 1.0
    ys = 2.0 * path_t[:, 1] / (h - 1) - 1.0
    grid = torch.stack([xs, ys], dim=-1).view(1, -1, 1, 2)
    sampled = F.grid_sample(image.unsqueeze(0), grid,
                            mode="bilinear", align_corners=True)
    signal = sampled.squeeze(-1).squeeze(0).T
    return (signal.clamp(0.0, 1.0) * beam_on_t.unsqueeze(-1))

In [ ]:
FLOWER_NAMES = [
    "pink primrose","hard-leaved pocket orchid","canterbury bells","sweet pea",
    "english marigold","tiger lily","moon orchid","bird of paradise","monkshood",
    "globe thistle","snapdragon","colts foot","king protea","spear thistle",
    "yellow iris","globe-flower","purple coneflower","peruvian lily",
    "balloon flower","giant white arum lily","fire lily","pincushion flower",
    "fritillary","red ginger","grape hyacinth","corn poppy",
    "prince of wales feathers","stemless gentian","artichoke","sweet william",
    "carnation","garden phlox","love in the mist","mexican aster",
    "alpine sea holly","ruby-lipped cattleya","cape flower","great masterwort",
    "siam tulip","lenten rose","barbeton daisy","daffodil","sword lily",
    "poinsettia","bolero deep blue","wallflower","marigold","buttercup",
    "oxeye daisy","common dandelion","petunia","wild pansy","primula",
    "sunflower","pelargonium","bishop of llandaff","gaura","geranium",
    "orange dahlia","pink-yellow dahlia","cautleya spicata","japanese anemone",
    "black-eyed susan","silverbush","californian poppy","osteospermum",
    "spring crocus","bearded iris","windflower","tree poppy","gazania",
    "azalea","water lily","rose","thorn apple","morning glory","passion flower",
    "lotus","toad lily","anthurium","frangipani","clematis","hibiscus",
    "columbine","desert-rose","tree mallow","magnolia","cyclamen","watercress",
    "canna lily","hippeastrum","bee balm","ball moss","foxglove","bougainvillea",
    "camellia","mallow","mexican petunia","bromelia","blanket flower",
    "trumpet creeper","blackberry lily",
]


class FlowerSignalDataset(Dataset):
    def __init__(self, hf_split, path_np, beam_on_np, image_size, label_names):
        from torchvision import transforms
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
        path_t = torch.from_numpy(path_np)
        beam_on_t = torch.from_numpy(beam_on_np.astype(np.float32))
        self.signals, self.captions = [], []
        for sample in hf_split:
            img = self.transform(sample["image"].convert("RGB"))
            sig = image_to_signal(img, path_t, beam_on_t)
            self.signals.append(sig.half())
            label = sample.get("label", 0)
            name = label_names[label] if label < len(label_names) else "flower"
            self.captions.append(f"a photo of a {name}")

    def __len__(self): return len(self.signals)
    def __getitem__(self, idx): return self.signals[idx].float(), self.captions[idx]

## 4. CLIP, positional encodings, transformer, head, model

In [ ]:
import clip


def load_clip(device):
    model, _ = clip.load("ViT-B/32", device=device)
    for p in model.parameters():
        p.requires_grad = False
    model.eval()
    return model


@torch.no_grad()
def encode_texts(clip_model, texts, device):
    tokens = clip.tokenize(texts, truncate=True).to(device)
    feats = clip_model.encode_text(tokens).float()
    return feats / feats.norm(dim=-1, keepdim=True)


def sinusoidal_1d(n_positions, dim):
    pe = torch.zeros(n_positions, dim)
    pos = torch.arange(n_positions, dtype=torch.float32).unsqueeze(1)
    div = torch.exp(torch.arange(0, dim, 2, dtype=torch.float32)
                    * -(math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


def sinusoidal_2d(coords, dim, max_coord):
    half = dim // 2
    div = torch.exp(torch.arange(0, half, 2, dtype=torch.float32)
                    * -(math.log(10000.0) / half))
    x = coords[:, 0:1] / max_coord * math.pi * 2 * (max_coord / 2)
    y = coords[:, 1:2] / max_coord * math.pi * 2 * (max_coord / 2)
    pe = torch.zeros(coords.shape[0], dim)
    pe[:, 0:half:2] = torch.sin(x * div)
    pe[:, 1:half:2] = torch.cos(x * div)
    pe[:, half::2] = torch.sin(y * div)
    pe[:, half + 1::2] = torch.cos(y * div)
    return pe


class FiLM(nn.Module):
    def __init__(self, cond_dim, feature_dim):
        super().__init__()
        self.to_scale_shift = nn.Linear(cond_dim, feature_dim * 2)
        nn.init.zeros_(self.to_scale_shift.weight)
        nn.init.zeros_(self.to_scale_shift.bias)

    def forward(self, x, cond):
        scale, shift = self.to_scale_shift(cond).chunk(2, dim=-1)
        return x * (1.0 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class CausalBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_mult, cond_dim, dropout, use_film):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.use_film = use_film
        self.norm1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, d_model * 3, bias=False)
        self.attn_out = nn.Linear(d_model, d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_model * ff_mult),
                                nn.GELU(),
                                nn.Linear(d_model * ff_mult, d_model))
        self.dropout = nn.Dropout(dropout)
        if use_film:
            self.film1 = FiLM(cond_dim, d_model)
            self.film2 = FiLM(cond_dim, d_model)

    def _attn(self, x):
        b, n, d = x.shape
        qkv = self.qkv(x).view(b, n, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.attn_out(out.transpose(1, 2).contiguous().view(b, n, d))

    def forward(self, x, cond):
        h = x + self.dropout(self._attn(self.norm1(x)))
        if self.use_film:
            h = self.film1(h, cond)
        h = h + self.dropout(self.ff(self.norm2(h)))
        if self.use_film:
            h = self.film2(h, cond)
        return h


class DMoLHead(nn.Module):
    def __init__(self, d_model, channels, n_mix):
        super().__init__()
        self.channels, self.n_mix = channels, n_mix
        self.out = nn.Linear(d_model, n_mix + channels * n_mix * 2)

    def forward(self, h):
        return self.out(h)

    def _split(self, params):
        k, c = self.n_mix, self.channels
        logit_probs = params[..., :k]
        rest = params[..., k:].view(*params.shape[:-1], c, k, 2)
        return logit_probs, rest[..., 0], rest[..., 1].clamp(min=-7.0)

    def nll(self, params, target):
        logit_probs, means, log_scales = self._split(params)
        t = target.unsqueeze(-1)
        inv_s = torch.exp(-log_scales)
        centered = t - means
        bin_half = 0.5 / 255.0
        plus_in = inv_s * (centered + bin_half)
        min_in = inv_s * (centered - bin_half)
        log_cdf_plus = plus_in - F.softplus(plus_in)
        log_one_minus_cdf_min = -F.softplus(min_in)
        prob = torch.sigmoid(plus_in) - torch.sigmoid(min_in)
        log_prob_mid = torch.log(prob.clamp(min=1e-12))
        log_probs = torch.where(
            target.unsqueeze(-1) < 1e-3, log_cdf_plus,
            torch.where(target.unsqueeze(-1) > 1.0 - 1e-3,
                        log_one_minus_cdf_min, log_prob_mid))
        log_probs = log_probs.sum(dim=-2)
        log_mix = F.log_softmax(logit_probs, dim=-1)
        return -torch.logsumexp(log_mix + log_probs, dim=-1)

    @torch.no_grad()
    def sample(self, params, temperature=1.0, top_p=1.0):
        logit_probs, means, log_scales = self._split(params)
        if temperature <= 0:
            k_idx = logit_probs.argmax(dim=-1)
        else:
            scaled = logit_probs / max(temperature, 1e-6)
            probs = F.softmax(scaled, dim=-1)
            if top_p < 1.0:
                sorted_p, sorted_i = probs.sort(dim=-1, descending=True)
                cum = sorted_p.cumsum(dim=-1)
                mask = cum - sorted_p > top_p
                sorted_p = sorted_p.masked_fill(mask, 0.0)
                sorted_p = sorted_p / sorted_p.sum(dim=-1, keepdim=True)
                probs = torch.zeros_like(probs).scatter_(-1, sorted_i, sorted_p)
            k_idx = torch.distributions.Categorical(probs=probs).sample()
        k_idx_exp = k_idx.unsqueeze(-1).unsqueeze(-1).expand(
            *k_idx.shape, self.channels, 1)
        chosen_mean = means.gather(-1, k_idx_exp).squeeze(-1)
        chosen_log_s = log_scales.gather(-1, k_idx_exp).squeeze(-1)
        if temperature <= 0:
            return chosen_mean.clamp(0.0, 1.0)
        u = torch.rand_like(chosen_mean).clamp(1e-5, 1.0 - 1e-5)
        sample = chosen_mean + torch.exp(chosen_log_s) * (
            torch.log(u) - torch.log1p(-u)) * temperature
        return sample.clamp(0.0, 1.0)


class SignalTransformer(nn.Module):
    def __init__(self, cfg, path, beam_on):
        super().__init__()
        self.cfg = cfg
        seq_len = len(path)
        input_ch = cfg.channels + (1 if cfg.use_flyback_mask else 0)
        self.input_proj = nn.Linear(input_ch, cfg.d_model)
        self.cond_proj = nn.Sequential(
            nn.Linear(cfg.clip_dim, cfg.cond_dim),
            nn.GELU(),
            nn.Linear(cfg.cond_dim, cfg.cond_dim))
        self.register_buffer("seq_pe", sinusoidal_1d(seq_len, cfg.d_model))
        path_t = torch.from_numpy(path)
        self.register_buffer("path_pe",
            sinusoidal_2d(path_t, cfg.d_model, max_coord=cfg.image_size - 1))
        self.register_buffer("beam_on_f",
            torch.from_numpy(beam_on.astype(np.float32)))
        self.blocks = nn.ModuleList([
            CausalBlock(cfg.d_model, cfg.n_heads, cfg.ff_mult,
                        cfg.cond_dim, cfg.dropout, cfg.use_film)
            for _ in range(cfg.n_layers)])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.head = DMoLHead(cfg.d_model, cfg.channels, cfg.n_mixtures)

    def _prepare_input(self, signal_prev):
        if self.cfg.use_flyback_mask:
            mask = self.beam_on_f[: signal_prev.size(1)]
            mask = mask.view(1, -1, 1).expand(signal_prev.size(0), -1, 1)
            return torch.cat([signal_prev, mask], dim=-1)
        return signal_prev

    def forward(self, shifted_signal, clip_emb):
        x = self._prepare_input(shifted_signal)
        b, n, _ = x.shape
        h = self.input_proj(x)
        if self.cfg.use_seq_pos_enc:
            h = h + self.seq_pe[:n].unsqueeze(0)
        if self.cfg.use_path_pos_enc:
            h = h + self.path_pe[:n].unsqueeze(0)
        cond = self.cond_proj(clip_emb)
        for block in self.blocks:
            h = block(h, cond)
        return self.head(self.norm_out(h))

    @torch.no_grad()
    def generate(self, clip_emb, n_steps, temperature=1.0, top_p=1.0):
        device = clip_emb.device
        b = clip_emb.size(0)
        c = self.cfg.channels
        cond = self.cond_proj(clip_emb)
        generated = torch.zeros(b, 0, c, device=device)
        prev = torch.zeros(b, 1, c, device=device)
        seq_pe = self.seq_pe.unsqueeze(0)
        path_pe = self.path_pe.unsqueeze(0)
        for i in range(n_steps):
            inp = torch.cat([prev, generated], dim=1) if generated.size(1) else prev
            x = self._prepare_input(inp)
            h = self.input_proj(x)
            if self.cfg.use_seq_pos_enc:
                h = h + seq_pe[:, : h.size(1)]
            if self.cfg.use_path_pos_enc:
                h = h + path_pe[:, : h.size(1)]
            for block in self.blocks:
                h = block(h, cond)
            params = self.head(self.norm_out(h)[:, -1:, :])
            next_sample = self.head.sample(params, temperature=temperature,
                                           top_p=top_p)
            if self.cfg.use_flyback_mask and not bool(self.beam_on_f[i]):
                next_sample = torch.zeros_like(next_sample)
            generated = torch.cat([generated, next_sample], dim=1)
        return generated

## 5. Training and evaluation (V5 protocol)

In [ ]:
LN2 = math.log(2.0)


@torch.no_grad()
def evaluate(model, clip_model, dataset, image_size, channels=3,
             batch_size=24, n_batches=20):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, drop_last=True)
    beam_on_f = model.beam_on_f
    total_nll = 0.0
    total_images = 0
    image_dims = image_size * image_size * channels
    for i, (signals, captions) in enumerate(loader):
        if i >= n_batches:
            break
        signals = signals.to(model.beam_on_f.device)
        b, n, c = signals.shape
        clip_emb = encode_texts(clip_model, list(captions),
                                model.beam_on_f.device)
        shifted = torch.cat(
            [torch.zeros(b, 1, c, device=signals.device),
             signals[:, :-1, :]], dim=1)
        params = model(shifted, clip_emb)
        nll = model.head.nll(params, signals)
        mask = beam_on_f.view(1, -1).expand(b, -1)
        total_nll += (nll * mask).sum().item()
        total_images += b
    mean_nll = total_nll / max(total_images, 1)
    return {
        "nll_per_image_nats": mean_nll,
        "bpd_pixel": mean_nll / LN2 / image_dims,
    }


def cosine_warmup_lr(step, warmup, total, base_lr):
    if step < warmup:
        return base_lr * step / max(warmup, 1)
    progress = (step - warmup) / max(total - warmup, 1)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def train(cfg, model, clip_model, dataset, val_dataset=None,
          ckpt_dir=None, log_every=5):
    from tqdm.auto import tqdm
    if ckpt_dir is not None:
        os.makedirs(ckpt_dir, exist_ok=True)
    loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True,
                        num_workers=0, drop_last=True, pin_memory=True)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                              weight_decay=cfg.weight_decay,
                              betas=(0.9, 0.95))
    total_steps = len(loader) * cfg.epochs
    beam_on_f = model.beam_on_f
    scaler = torch.amp.GradScaler(cfg.device) if cfg.device == "cuda" else None
    step = 0
    val_history = []
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        running, running_n = 0.0, 0
        pbar = tqdm(loader, desc=f"epoch {epoch}/{cfg.epochs}")
        for signals, captions in pbar:
            signals = signals.to(cfg.device, non_blocking=True)
            b, n, c = signals.shape
            with torch.no_grad():
                clip_emb = encode_texts(clip_model, list(captions), cfg.device)
            p_ss = cfg.ss_max * min(step / max(total_steps, 1), 1.0)
            shifted = torch.cat(
                [torch.zeros(b, 1, c, device=cfg.device),
                 signals[:, :-1, :]], dim=1)
            if p_ss > 0:
                with torch.no_grad():
                    preview_params = model(shifted, clip_emb)
                    preview = model.head.sample(preview_params, temperature=1.0)
                    mask_ss = (torch.rand(b, n, 1, device=cfg.device)
                               < p_ss).float()
                    mixed = shifted.clone()
                    mixed[:, 1:, :] = (
                        mask_ss[:, 1:, :] * preview[:, :-1, :]
                        + (1.0 - mask_ss[:, 1:, :]) * shifted[:, 1:, :])
                    shifted = mixed
            lr = cosine_warmup_lr(step, cfg.warmup_steps, total_steps, cfg.lr)
            for g in optim.param_groups:
                g["lr"] = lr
            optim.zero_grad(set_to_none=True)
            if scaler is not None:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    params = model(shifted, clip_emb)
                    nll = model.head.nll(params, signals)
                    mask = beam_on_f.view(1, -1).expand(b, -1)
                    loss = (nll * mask).sum() / mask.sum().clamp(min=1.0)
                scaler.scale(loss).backward()
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optim)
                scaler.update()
            else:
                params = model(shifted, clip_emb)
                nll = model.head.nll(params, signals)
                mask = beam_on_f.view(1, -1).expand(b, -1)
                loss = (nll * mask).sum() / mask.sum().clamp(min=1.0)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                optim.step()
            running += loss.item() * b
            running_n += b
            step += 1
            pbar.set_postfix(nll=f"{loss.item():.3f}", lr=f"{lr:.2e}")
        avg = running / running_n
        print(f"epoch {epoch}/{cfg.epochs} avg_nll={avg:.4f}")
        if val_dataset is not None and epoch % log_every == 0:
            val_m = evaluate(model, clip_model, val_dataset,
                             image_size=cfg.image_size,
                             batch_size=cfg.batch_size)
            val_history.append({"epoch": epoch, **val_m})
            print(f"  val bpd_pixel={val_m['bpd_pixel']:.4f}")
            if ckpt_dir is not None:
                torch.save({"model": model.state_dict(), "epoch": epoch,
                            "val_bpd_pixel": val_m["bpd_pixel"]},
                           f"{ckpt_dir}/epoch_{epoch}.pt")
    return val_history

## 6. Load CLIP and Oxford Flowers (used by all priorities)

In [ ]:
clip_model = load_clip(cfg.device)

In [ ]:
from datasets import load_dataset
hf_train_full = load_dataset("nelorth/oxford-flowers", split="train")
hf_test_full  = load_dataset("nelorth/oxford-flowers", split="test")
print(f"train: {len(hf_train_full)}  test: {len(hf_test_full)}")

## 7. PRIORITY 1: Multi-seed verification at 48x48

The paper's scaling claim (Section IV-C, Table II) currently relies
on a single seed. We replace that with a 3-seed mean and standard
deviation by running both `full` and `path-only` configurations at
$48\times 48$ for 15 epochs, 3 seeds each.

**Compute budget:** 6 runs at ~1 hour each = ~6 hours T4.

**Expected outcome (from previous single-seed run):**
- full: bpd_pixel = 8.681
- path only: bpd_pixel = 7.220
- delta = 1.46 bpd in favor of path-only

If the multi-seed mean stays close to these numbers and the std is
moderate (< 0.3 bpd), the paper's scaling claim becomes a robust
multi-seed result instead of an anecdote.

Set `RUN_P1 = True` and execute the cells below.

In [ ]:
# ── P1 split: run P1_A first (full, 3 seeds), then P1_B (path_only, 3 seeds) ──
# Set ONE flag to True per Kaggle session; leave the other False.
RUN_P1_A = False   # trains: 48x48 full      seeds 0,1,2  (~6 h T4)
RUN_P1_B = False   # trains: 48x48 path_only seeds 0,1,2  (~6 h T4)

if RUN_P1_A or RUN_P1_B:
    cfg48 = copy.deepcopy(cfg)
    cfg48.image_size = 48
    cfg48.epochs = 15  # ~1h each on T4

    path48_np, beam_on48_np = generate_raster_path(
        cfg48.image_size, cfg48.image_size,
        cfg48.samples_per_pixel, cfg48.flyback_frac)
    SEQ_LEN_48 = len(path48_np)
    print(f"48x48 seq_len: {SEQ_LEN_48}")

    dataset48 = FlowerSignalDataset(
        hf_train_full, path48_np, beam_on48_np,
        cfg48.image_size, FLOWER_NAMES)
    testset48 = FlowerSignalDataset(
        hf_test_full, path48_np, beam_on48_np,
        cfg48.image_size, FLOWER_NAMES)
    print(f"train48: {len(dataset48)}  test48: {len(testset48)}")
else:
    print("RUN_P1_A and RUN_P1_B are both False - skipping setup.")


In [ ]:
def run_one_48(use_seq_pe: bool, seed: int, epochs: int = 15):
    run_cfg = copy.deepcopy(cfg48)
    run_cfg.use_seq_pos_enc = use_seq_pe
    run_cfg.epochs = epochs
    torch.manual_seed(seed)
    np.random.seed(seed)
    m = SignalTransformer(run_cfg, path48_np, beam_on48_np).to(run_cfg.device)
    name = "full" if use_seq_pe else "path_only"
    ckpt_dir = f"{WORK_DIR}/ckpt_48_{name}_seed{seed}"
    train(run_cfg, m, clip_model, dataset48,
          val_dataset=testset48, ckpt_dir=ckpt_dir, log_every=5)
    metrics = evaluate(m, clip_model, testset48,
                       image_size=run_cfg.image_size,
                       batch_size=run_cfg.batch_size)
    return metrics


In [ ]:
# ── P1_A: 48x48 "full" (both PEs on), 3 seeds ──────────────────────────────
# After this cell finishes, copy the 3 printed bpd_pixel values into
# P1A_FULL_BPDS at the top of the P1_B cell below.
import json as _json

if RUN_P1_A:
    p1a_results = []
    for seed in (0, 1, 2):
        print(f"\n=== 48x48 full seed={seed} ===")
        m = run_one_48(use_seq_pe=True, seed=seed, epochs=cfg48.epochs)
        p1a_results.append(m)
        print(f"  bpd_pixel = {m['bpd_pixel']:.4f}  "
              f"nll = {m['nll_per_image_nats']:.1f}")

    print("\n" + "=" * 50)
    print("P1_A DONE – copy these 3 bpd_pixel values into P1_B:")
    for i, r in enumerate(p1a_results):
        print(f"  seed {i}: bpd_pixel = {r['bpd_pixel']:.4f}")

    # Save to disk so P1_B can load them if run in the same session
    _out = f"{WORK_DIR}/p1a_results.json"
    _json.dump(p1a_results, open(_out, "w"))
    print(f"  (also saved to {_out})")
else:
    print("RUN_P1_A is False – skipped.")


In [ ]:
# ── P1_B: 48x48 "path_only" (no seq PE), 3 seeds + aggregate ───────────────
#
# BEFORE running: fill P1A_FULL_BPDS and P1A_FULL_NLLS with the numbers
# printed by P1_A (or leave as empty lists if running in the SAME session
# that ran P1_A – they will be loaded from p1a_results.json automatically).
import json as _json, os as _os

P1A_FULL_BPDS = []   # ← paste here, e.g. [8.8166, 8.8464, 8.7944]
P1A_FULL_NLLS = []   # ← paste here, e.g. [42240.5, 42383.3, 42134.3]

if RUN_P1_B:
    # Try loading P1_A results from disk (same session) or from hardcoded values
    _p1a_path = f"{WORK_DIR}/p1a_results.json"
    if not P1A_FULL_BPDS and _os.path.exists(_p1a_path):
        _loaded = _json.load(open(_p1a_path))
        P1A_FULL_BPDS = [r["bpd_pixel"] for r in _loaded]
        P1A_FULL_NLLS = [r["nll_per_image_nats"] for r in _loaded]
        print(f"Loaded P1_A results from {_p1a_path}")
    elif not P1A_FULL_BPDS:
        print("WARNING: P1A_FULL_BPDS is empty and no p1a_results.json found.")
        print("Fill P1A_FULL_BPDS / P1A_FULL_NLLS manually before computing aggregate.")

    p1b_results = []
    for seed in (0, 1, 2):
        print(f"\n=== 48x48 path_only seed={seed} ===")
        m = run_one_48(use_seq_pe=False, seed=seed, epochs=cfg48.epochs)
        p1b_results.append(m)
        print(f"  bpd_pixel = {m['bpd_pixel']:.4f}  "
              f"nll = {m['nll_per_image_nats']:.1f}")

    # ── Aggregate ────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("PRIORITY 1 FINAL: 48x48 multi-seed (3 seeds, 15 epochs)")
    print("=" * 60)

    po_bpds = [r["bpd_pixel"] for r in p1b_results]
    po_nlls = [r["nll_per_image_nats"] for r in p1b_results]

    for label, bpds, nlls in [
        ("full (P1_A)",       P1A_FULL_BPDS, P1A_FULL_NLLS),
        ("path_only (P1_B)",  po_bpds,       po_nlls),
    ]:
        if len(bpds) < 2:
            print(f"\n{label}: insufficient data ({len(bpds)} seeds)")
            continue
        print(f"\n{label}:")
        print(f"  bpd_pixel:  {statistics.mean(bpds):.4f}  +/-  {statistics.stdev(bpds):.4f}")
        print(f"  nll/image:  {statistics.mean(nlls):.1f}  +/-  {statistics.stdev(nlls):.1f}")

    if len(P1A_FULL_NLLS) == 3 and len(po_nlls) == 3:
        delta = statistics.mean(po_nlls) - statistics.mean(P1A_FULL_NLLS)
        se = math.sqrt(statistics.stdev(P1A_FULL_NLLS) ** 2 / 3
                       + statistics.stdev(po_nlls) ** 2 / 3)
        t = delta / se if se > 0 else float("nan")
        print(f"\nWelch t-test (path_only - full):")
        print(f"  delta nll = {delta:+.1f}  (negative => path_only wins)")
        print(f"  SE = {se:.1f}, t = {t:+.2f}")
    else:
        print("\n(Aggregate skipped: need 3 seeds from both P1_A and P1_B)")
else:
    print("RUN_P1_B is False – skipped.")


## 8. PRIORITY 2: Generation samples for paper figure

Two side-by-side grids of 6 samples each at temperature 1.0, top_p
0.95. Save as PNGs at high resolution for inclusion in the paper.

Requires the V5 path-only checkpoint at epoch 55 (the best
checkpoint from the 60-epoch convergence run reported in Section
IV-B). If the checkpoint is not available, the code below trains a
fresh path-only model for 60 epochs to produce one. Comment out the
`fresh_train` branch if you have an existing checkpoint and load it
manually.

For PixelCNN++ samples, the same approach: load existing checkpoint
or train a fresh PixelCNN++ for 60 epochs (~4h).

**Compute budget if no checkpoints available:**
- V5 path-only training: ~4 hours T4
- PixelCNN++ training: ~4 hours T4
- Generation: ~1 hour total (V5 is fast, PixelCNN++ is O(H*W))

**Compute budget if checkpoints exist:** ~1 hour total.

In [ ]:
RUN_P2 = False  # set to True to run

# ── Checkpoint paths for P2 ─────────────────────────────────────────────────
# P2 needs two checkpoints from EARLIER notebooks (not from P1_A/P1_B).
# These checkpoints were trained on 32x32 images.
#
# Checkpoint 1 — V5 path-only, 60 epochs (no_seq_pe):
#   Source: your V5 Part-1 training notebook (signal-space, 32x32, 60-epoch run)
#   Kaggle: go to that notebook → Output tab → "Add as Input" to this notebook
#   Input dataset name becomes something like /kaggle/input/<that-notebook-slug>/
#
# Checkpoint 2 — PixelCNN++ baseline, 60 epochs:
#   Source: pixelcnn_baseline_flowers notebook (Section 10)
#   Kaggle: same process → Add as Input
#
# After adding them as Inputs, set the two paths below to match exactly.
# If you leave the defaults and the files are missing, P2 will:
#   - V5:      train a fresh 60-epoch model (~4 h) then generate  ← slow but works
#   - PixelCNN: skip its samples and skip the comparison figure   ← no crash

def _find_ckpt(*candidates):
    """Return first existing path, or None."""
    import os
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return None

import glob as _glob

# Auto-search for V5 checkpoint.
# Part-3 experiments notebook saves to: checkpoints_no_seq_pe/
# Fallback names also covered: ckpt_v5_path_only*, checkpoints_v5_path*
_v5_search_55 = (
    sorted(_glob.glob("/kaggle/input/**/checkpoints_no_seq_pe/**/epoch_55.pt", recursive=True)) or
    sorted(_glob.glob("/kaggle/input/**/ckpt_v5_path_only*/**/epoch_55.pt",    recursive=True)) or
    sorted(_glob.glob("/kaggle/input/**/checkpoints_v5*/**/epoch_55.pt",       recursive=True))
)
_v5_search_60 = (
    sorted(_glob.glob("/kaggle/input/**/checkpoints_no_seq_pe/**/epoch_60.pt", recursive=True)) or
    sorted(_glob.glob("/kaggle/input/**/ckpt_v5_path_only*/**/epoch_60.pt",    recursive=True))
)
_v5_search_best = sorted(_glob.glob("/kaggle/input/**/checkpoints_no_seq_pe/**/best.pt", recursive=True))
_v5_search = _v5_search_55 or _v5_search_60 or _v5_search_best  # prefer epoch_55, then 60, then best
# Search for epoch_60.pt OR best.pt under any checkpoints_pixelcnn* folder
_pcnn_search_60 = sorted(_glob.glob("/kaggle/input/**/checkpoints_pixelcnn*/**/epoch_60.pt", recursive=True))
_pcnn_search_best = sorted(_glob.glob("/kaggle/input/**/checkpoints_pixelcnn*/**/best.pt", recursive=True))
_pcnn_search = _pcnn_search_60 or _pcnn_search_best  # prefer 60-ep, fall back to best.pt

# ── Set these manually if auto-search fails ──────────────────────────────
# Dataset: assemelqirsh/paper-checkpoints (add via + icon → Datasets)
# Contains: crt_epoch_60.pt  (V5 no_seq_pe 60-epoch)
#           pixelcnn_epoch_60.pt  (PixelCNN++ 60-epoch)
V5_CKPT_PATH_OVERRIDE   = "/kaggle/input/datasets/assemelqirsh/paper-checkpoints/crt_epoch_60.pt"
# ↓ EXACT PATH from paper-checkpoints dataset:
PCNN_CKPT_PATH_OVERRIDE = "/kaggle/input/datasets/assemelqirsh/paper-checkpoints/pixelcnn_epoch_60.pt"

V5_CKPT_PATH = _find_ckpt(
    V5_CKPT_PATH_OVERRIDE or None,
    _v5_search[0] if _v5_search else None,
    f"{WORK_DIR}/checkpoints_no_seq_pe/epoch_55.pt",
    f"{WORK_DIR}/checkpoints_no_seq_pe/epoch_60.pt",
    f"{WORK_DIR}/ckpt_v5_path_only_60ep/epoch_55.pt",
)
PCNN_CKPT_PATH = _find_ckpt(
    PCNN_CKPT_PATH_OVERRIDE or None,
    _pcnn_search[0] if _pcnn_search else None,
    f"{WORK_DIR}/checkpoints_pixelcnn/best.pt",
    f"{WORK_DIR}/ckpt_pixelcnn_60ep/epoch_60.pt",
)

print(f"V5   checkpoint : {V5_CKPT_PATH   or 'NOT FOUND – will train from scratch (~4 h)'}")
print(f"PCNN checkpoint : {PCNN_CKPT_PATH or 'NOT FOUND – PixelCNN++ samples will be skipped'}")

PROMPTS = [
    "a photo of a red rose",
    "a photo of a yellow sunflower",
    "a photo of a purple iris",
    "a photo of a white lily",
    "a photo of a pink daisy",
    "a photo of an orange marigold",
]

if RUN_P2:
    # 32x32 path setup (matches V5 at 60 epochs)
    path32_np, beam_on32_np = generate_raster_path(
        cfg.image_size, cfg.image_size,
        cfg.samples_per_pixel, cfg.flyback_frac)
    SEQ_LEN = len(path32_np)
    renderer32 = CRTRenderer(path32_np, beam_on32_np,
                             cfg.image_size, cfg.beam_sigma).to(cfg.device)

    # Load or train V5 path-only model
    v5_cfg = copy.deepcopy(cfg)
    v5_cfg.use_seq_pos_enc = False
    v5_cfg.epochs = 60
    v5_model = SignalTransformer(v5_cfg, path32_np, beam_on32_np).to(cfg.device)

    if V5_CKPT_PATH:
        print(f"Loading V5 checkpoint from {V5_CKPT_PATH}")
        ckpt = torch.load(V5_CKPT_PATH, map_location=cfg.device)
        v5_model.load_state_dict(ckpt["model"])
    else:
        print("V5 checkpoint not found; training fresh path-only model "
              "for 60 epochs (~4h on T4)")
        dataset32 = FlowerSignalDataset(hf_train_full, path32_np, beam_on32_np,
                                        cfg.image_size, FLOWER_NAMES)
        testset32 = FlowerSignalDataset(hf_test_full, path32_np, beam_on32_np,
                                        cfg.image_size, FLOWER_NAMES)
        train(v5_cfg, v5_model, clip_model, dataset32,
              val_dataset=testset32,
              ckpt_dir=f"{WORK_DIR}/ckpt_v5_path_only_60ep")
else:
    print("RUN_P2 is False - skipping. Set to True and re-run to enable.")


In [ ]:
@torch.no_grad()
def generate_v5_samples(model, renderer, clip_model, prompts,
                        n_steps, temperature=1.0, top_p=0.95):
    model.eval()
    clip_emb = encode_texts(clip_model, prompts, cfg.device)
    signals = model.generate(clip_emb, n_steps, temperature=temperature,
                             top_p=top_p)
    images = renderer(signals).cpu()
    return images


def save_grid(images, prompts, save_path, title=None):
    n = len(prompts)
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.5))
    if n == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        img = images[i].permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img)
        # Strip the "a photo of a" prefix for compactness
        short = prompts[i].replace("a photo of a ", "").replace("a photo of an ", "")
        ax.set_title(short, fontsize=8)
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=10, y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"saved -> {save_path}")


if RUN_P2:
    print("Generating V5 path-only samples...")
    v5_imgs = generate_v5_samples(v5_model, renderer32, clip_model,
                                  PROMPTS, SEQ_LEN,
                                  temperature=1.0, top_p=0.95)
    save_grid(v5_imgs, PROMPTS,
              f"{WORK_DIR}/v5_path_only_samples.png",
              title="Signal-space (path only, V5, epoch 55)")

### PixelCNN++ samples

The PixelCNN++ class definition is included here so this notebook is
self-contained for sample generation.

In [ ]:
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        assert mask_type in ("A", "B")
        _, _, kh, kw = self.weight.shape
        mask = torch.ones(kh, kw)
        mask[kh // 2 + 1:, :] = 0.0
        mask[kh // 2, kw // 2 + 1:] = 0.0
        if mask_type == "A":
            mask[kh // 2, kw // 2] = 0.0
        self.register_buffer("mask", mask.view(1, 1, kh, kw))

    def forward(self, x):
        self.weight.data.mul_(self.mask)
        return super().forward(x)


class FiLM2D(nn.Module):
    def __init__(self, cond_dim, feature_dim):
        super().__init__()
        self.to_scale_shift = nn.Linear(cond_dim, feature_dim * 2)
        nn.init.zeros_(self.to_scale_shift.weight)
        nn.init.zeros_(self.to_scale_shift.bias)

    def forward(self, x, cond):
        scale, shift = self.to_scale_shift(cond).chunk(2, dim=-1)
        return x * (1.0 + scale.unsqueeze(-1).unsqueeze(-1)) +                shift.unsqueeze(-1).unsqueeze(-1)


class GatedMaskedResBlock(nn.Module):
    def __init__(self, n_filters, cond_dim, dropout):
        super().__init__()
        self.conv1 = MaskedConv2d("B", n_filters, n_filters,
                                  kernel_size=3, padding=1)
        self.conv2 = MaskedConv2d("B", n_filters, 2 * n_filters,
                                  kernel_size=3, padding=1)
        self.film = FiLM2D(cond_dim, n_filters)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, cond):
        h = F.elu(x)
        h = self.conv1(h)
        h = self.film(h, cond)
        h = self.dropout(F.elu(h))
        h = self.conv2(h)
        a, b = h.chunk(2, dim=1)
        return x + a * torch.sigmoid(b)


class DMoLHead2D(nn.Module):
    def __init__(self, n_filters, channels, n_mix):
        super().__init__()
        self.channels, self.n_mix = channels, n_mix
        self.proj = MaskedConv2d("B", n_filters,
                                 n_mix + channels * n_mix * 2, kernel_size=1)

    def forward(self, h):
        return self.proj(h).permute(0, 2, 3, 1).contiguous()

    def _split(self, params):
        k, c = self.n_mix, self.channels
        logit_probs = params[..., :k]
        rest = params[..., k:].view(*params.shape[:-1], c, k, 2)
        return logit_probs, rest[..., 0], rest[..., 1].clamp(min=-7.0)

    @torch.no_grad()
    def sample(self, params, temperature=1.0):
        logit_probs, means, log_scales = self._split(params)
        if temperature <= 0:
            k_idx = logit_probs.argmax(dim=-1)
        else:
            probs = F.softmax(logit_probs / max(temperature, 1e-6), dim=-1)
            k_idx = torch.distributions.Categorical(probs=probs).sample()
        k_idx_exp = k_idx.unsqueeze(-1).unsqueeze(-1).expand(
            *k_idx.shape, self.channels, 1)
        chosen_mean = means.gather(-1, k_idx_exp).squeeze(-1)
        chosen_log_s = log_scales.gather(-1, k_idx_exp).squeeze(-1)
        if temperature <= 0:
            return chosen_mean.clamp(0.0, 1.0)
        u = torch.rand_like(chosen_mean).clamp(1e-5, 1.0 - 1e-5)
        sample = chosen_mean + torch.exp(chosen_log_s) * (
            torch.log(u) - torch.log1p(-u)) * temperature
        return sample.clamp(0.0, 1.0)


@dataclass
class PCNNConfig:
    image_size: int = 32
    channels: int = 3
    clip_dim: int = 512
    cond_dim: int = 256
    n_filters: int = 160
    n_resnet: int = 8
    n_mixtures: int = 5
    dropout: float = 0.1
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class PixelCNNpp(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.cond_proj = nn.Sequential(
            nn.Linear(cfg.clip_dim, cfg.cond_dim),
            nn.GELU(),
            nn.Linear(cfg.cond_dim, cfg.cond_dim))
        self.input_conv = MaskedConv2d("A", cfg.channels, cfg.n_filters,
                                       kernel_size=7, padding=3)
        self.blocks = nn.ModuleList([
            GatedMaskedResBlock(cfg.n_filters, cfg.cond_dim, cfg.dropout)
            for _ in range(cfg.n_resnet)])
        self.head = DMoLHead2D(cfg.n_filters, cfg.channels, cfg.n_mixtures)

    def forward(self, image, clip_emb):
        cond = self.cond_proj(clip_emb)
        h = self.input_conv(image)
        for block in self.blocks:
            h = block(h, cond)
        return self.head(h)

    @torch.no_grad()
    def generate(self, clip_emb, image_size, temperature=1.0):
        device = clip_emb.device
        b = clip_emb.size(0)
        c = self.cfg.channels
        img = torch.zeros(b, c, image_size, image_size, device=device)
        for i in range(image_size):
            for j in range(image_size):
                params = self(img, clip_emb)
                pred = self.head.sample(
                    params[:, i:i+1, j:j+1, :], temperature=temperature)
                img[:, :, i, j] = pred[:, 0, 0, :]
        return img

In [ ]:
if RUN_P2:
    pcnn_cfg = PCNNConfig()
    pcnn_model = PixelCNNpp(pcnn_cfg).to(pcnn_cfg.device)

    if os.path.exists(PCNN_CKPT_PATH):
        print(f"Loading PixelCNN++ checkpoint from {PCNN_CKPT_PATH}")
        ckpt = torch.load(PCNN_CKPT_PATH, map_location=pcnn_cfg.device)
        pcnn_model.load_state_dict(ckpt["model"])
    else:
        print("PixelCNN++ checkpoint not found.")
        print("Re-run the PixelCNN++ baseline notebook to obtain it,")
        print("or use the V5 60-epoch checkpoint alone (skip this cell).")
        pcnn_model = None

    if pcnn_model is not None:
        print("\nGenerating PixelCNN++ samples (this takes ~10 minutes)...")
        pcnn_model.eval()
        with torch.no_grad():
            clip_emb = encode_texts(clip_model, PROMPTS, pcnn_cfg.device)
            pcnn_imgs = pcnn_model.generate(
                clip_emb, pcnn_cfg.image_size, temperature=1.0).cpu()
        save_grid(pcnn_imgs, PROMPTS,
                  f"{WORK_DIR}/pixelcnn_samples.png",
                  title="Pixel-space (PixelCNN++, epoch 60)")

### Side-by-side comparison figure

If both V5 and PixelCNN++ samples were generated above, this cell
combines them into a single figure for the paper.

In [ ]:
if RUN_P2 and 'v5_imgs' in dir() and 'pcnn_imgs' in dir():
    n = len(PROMPTS)
    fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 5.5))
    for i in range(n):
        # Top row: V5 path-only
        img_v5 = v5_imgs[i].permute(1, 2, 0).clamp(0, 1).numpy()
        axes[0, i].imshow(img_v5)
        axes[0, i].axis("off")
        short = PROMPTS[i].replace("a photo of a ", "").replace(
            "a photo of an ", "")
        axes[0, i].set_title(short, fontsize=9)
        # Bottom row: PixelCNN++
        img_pcnn = pcnn_imgs[i].permute(1, 2, 0).clamp(0, 1).numpy()
        axes[1, i].imshow(img_pcnn)
        axes[1, i].axis("off")
    # Row labels
    axes[0, 0].text(-0.15, 0.5, "Signal\nspace",
                    transform=axes[0, 0].transAxes,
                    rotation=90, va="center", ha="center", fontsize=10)
    axes[1, 0].text(-0.15, 0.5, "Pixel\nspace",
                    transform=axes[1, 0].transAxes,
                    rotation=90, va="center", ha="center", fontsize=10)
    plt.tight_layout()
    out_path = f"{WORK_DIR}/comparison_figure.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"\nside-by-side figure saved -> {out_path}")
    print("Insert this in the paper as Figure 3.")

## 9. PRIORITY 3: Complete the 32x32 ablation grid

The paper's Table I marks `seq only` and `neither` as "single seed
only". This section runs both with 3 seeds at 30 epochs to complete
the 4-cell ablation matrix.

**Compute budget:** 6 runs at ~30 minutes each = ~3 hours T4.

**Outcome:** A complete Table I with multi-seed mean +/- std for all
four configurations: full, seq only, path only, neither.

In [ ]:
RUN_P3 = False  # set to True to run

if RUN_P3:
    cfg32 = copy.deepcopy(cfg)
    cfg32.epochs = 30

    path32_np, beam_on32_np = generate_raster_path(
        cfg32.image_size, cfg32.image_size,
        cfg32.samples_per_pixel, cfg32.flyback_frac)

    dataset32 = FlowerSignalDataset(hf_train_full, path32_np, beam_on32_np,
                                    cfg32.image_size, FLOWER_NAMES)
    testset32 = FlowerSignalDataset(hf_test_full, path32_np, beam_on32_np,
                                    cfg32.image_size, FLOWER_NAMES)
else:
    print("RUN_P3 is False - skipping. Set to True and re-run to enable.")

In [ ]:
def run_one_32(use_seq_pos_enc: bool, use_path_pos_enc: bool, seed: int,
               epochs: int = 30):
    run_cfg = copy.deepcopy(cfg32)
    run_cfg.use_seq_pos_enc = use_seq_pos_enc
    run_cfg.use_path_pos_enc = use_path_pos_enc
    run_cfg.epochs = epochs
    torch.manual_seed(seed)
    np.random.seed(seed)
    m = SignalTransformer(run_cfg, path32_np, beam_on32_np).to(run_cfg.device)
    label = ("seq" if use_seq_pos_enc else "noseq") + "_" + (
        "path" if use_path_pos_enc else "nopath")
    train(run_cfg, m, clip_model, dataset32,
          val_dataset=testset32,
          ckpt_dir=f"{WORK_DIR}/ckpt_32_{label}_seed{seed}",
          log_every=10)
    return evaluate(m, clip_model, testset32,
                    image_size=run_cfg.image_size,
                    batch_size=run_cfg.batch_size)


if RUN_P3:
    # Only the two missing configurations: seq only, neither
    grid = [
        ("seq_only", {"use_seq_pos_enc": True,  "use_path_pos_enc": False}),
        ("neither",  {"use_seq_pos_enc": False, "use_path_pos_enc": False}),
    ]
    p3_results = {}
    for name, kw in grid:
        seeds_results = []
        for seed in (0, 1, 2):
            print(f"\n=== {name} seed={seed} ===")
            m = run_one_32(seed=seed, epochs=cfg32.epochs, **kw)
            seeds_results.append(m)
            print(f"  bpd_pixel = {m['bpd_pixel']:.4f}")
        p3_results[name] = seeds_results

    print("\n" + "=" * 60)
    print("PRIORITY 3 FINAL: Complete 32x32 grid (3 seeds, 30 epochs)")
    print("=" * 60)
    for name, results in p3_results.items():
        bpds = [r["bpd_pixel"] for r in results]
        nlls = [r["nll_per_image_nats"] for r in results]
        print(f"\n{name}:")
        print(f"  bpd_pixel:  {statistics.mean(bpds):.4f}"
              f" +/- {statistics.stdev(bpds):.4f}")
        print(f"  nll/image:  {statistics.mean(nlls):.1f}"
              f" +/- {statistics.stdev(nlls):.1f}")

## 10. After running: updating the paper

Once these runs complete, the paper updates are:

**P1 -> Table II.** Replace the single-seed row with multi-seed
numbers. The current Table II is:

```
Configuration    bpd
seq + path       8.681
path only        7.220
delta            1.461
```

After P1, replace with:
```
Configuration    bpd
seq + path       <P1 full mean>  +/- <std>
path only        <P1 path_only mean> +/- <std>
delta + t        <delta>  (t = <t-statistic>)
```

**P2 -> Figure 3 (NEW).** Add `comparison_figure.png` as Figure 3
in the Experiments section right after the PixelCNN++ comparison
table. Suggested caption: "Generation samples at temperature 1.0,
top_p 0.95. Top: signal-space model (path only, epoch 55).
Bottom: matched-parameter PixelCNN++ (epoch 60). Both rows use
identical CLIP-conditioned prompts."

**P3 -> Table I.** Replace the "single seed only" notes for
`seq only` and `neither` with multi-seed numbers. The footnote
about not allocating full multi-seed budget is no longer needed.

If only P1 is run, the paper already has the most important update.
P2 is the most paper-improving among the remaining; P3 is the
nice-to-have completion.